<a href="https://colab.research.google.com/github/Justin21523/edge-deid-studio/blob/feature%2Fner-pipeline/notebooks/07_ner_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab 第一格：掛載 Google Drive、設定快取，安裝必要套件
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 把 HF 模型與資料快取都放到 Drive，避免重啟就消失
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_home'
os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/hf_datasets'
os.environ['HF_HUB_CACHE'] = '/content/drive/MyDrive/hf_hub'

!pip install -q transformers datasets accelerate seqeval onnx onnxruntime onnxruntime-tools torch


In [ ]:
# Colab 第二格：載入 PII 資料集與 CLUENER2020（xusenlin/clue-ner）
from datasets import load_dataset

# 1) ai4privacy/pii-masking-200k：20 萬筆中英文混合 PII 遮蔽語料
pii_ds = load_dataset("ai4privacy/pii-masking-200k", split="train")   # 共約 209k 筆 :contentReference[oaicite:0]{index=0}

# 2) xusenlin/clue-ner：13k 筆細粒度中文 NER，10 類標註 （person, org, address…）:contentReference[oaicite:1]{index=1}
cluener = load_dataset("xusenlin/clue-ner", split="train")

# 簡單檢視前兩筆
print(pii_ds[0])
print(cluener[0])


In [ ]:
# Colab 第三格：Tokenizer 與標籤映射
from transformers import AutoTokenizer

# 文字 NER 模型：以 ckiplab/bert-base-chinese-ner 作為基底
tokenizer = AutoTokenizer.from_pretrained("ckiplab/bert-base-chinese-ner", use_fast=True)

# 準備 label list（從 cluener dataset 取得）
labels = cluener.features['entities']['label'].names if 'entities' in cluener.features else ["O","PER","LOC","ORG","TIME",...]

# 建立映射字典
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

# 處理函式：tokenize 並對齊子詞標籤
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    word_ids = tokenized.word_ids()
    aligned_labels = []
    for i, word_id in enumerate(word_ids):
        if word_id is None:
            aligned_labels.append(-100)
        else:
            entity = examples['entities'][i]  # 根據 cluener 結構自行調整取值
            aligned_labels.append(label2id.get(entity, 0) if word_id==0 else -100)
    tokenized["labels"] = aligned_labels
    return tokenized

# 將 cluener 與 pii-masking 合併微調：這裡示範只用 cluener，實際可 concat
train_ds = cluener.map(tokenize_and_align_labels, batched=True)


In [ ]:
# Colab 第四格：建立 Trainer，進行中文 NER 微調
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 載入模型
model = AutoModelForTokenClassification.from_pretrained(
    "ckiplab/bert-base-chinese-ner",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# seqeval 指標
metric = evaluate.load("seqeval")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    true = p.label_ids
    # 把 -100 忽略
    true_labels = [[id2label[l] for l in row if l != -100] for row in true]
    pred_labels = [[id2label[p] for (p, l) in zip(row_pred, row_true) if l != -100]
                   for row_pred, row_true in zip(preds, true)]
    results = metric.compute(predictions=pred_labels, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results.get("overall_accuracy", 0)
    }

# 訓練參數
args = TrainingArguments(
    output_dir="drive/MyDrive/edge-deid/models/ner/v1.0",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=train_ds,   # demo 用；實務上要分 split
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 真正啟動訓練
trainer.train()
trainer.save_model()      # 存到 Drive


In [ ]:
# Colab 第五格：ONNX 匯出與動態 Int8 量化
import torch, onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

# 載入剛剛訓練好的 PyTorch 模型
model = AutoModelForTokenClassification.from_pretrained("drive/MyDrive/edge-deid/models/ner/v1.0")
model.eval()

# 隨機輸入示例
dummy_input = tokenizer("測試文字", return_tensors="pt", padding="max_length", max_length=128)
torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    "drive/MyDrive/edge-deid/onnx/ner.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
)

# 動態量化
quantize_dynamic(
    "drive/MyDrive/edge-deid/onnx/ner.onnx",
    "drive/MyDrive/edge-deid/onnx/ner_int8.onnx",
    weight_type=QuantType.QInt8,
)


In [ ]:
# Colab 第六格：對測試集做評估並視覺化 F1 結果
# 重新載入 ONNX 模型測試推論（示範單例）
ort_sess = onnxruntime.InferenceSession("drive/MyDrive/edge-deid/onnx/ner_int8.onnx")
inputs  = {k: v.cpu().numpy() for k,v in dummy_input.items()}
logits  = ort_sess.run(None, inputs)[0]
pred_ids = np.argmax(logits, axis=-1)[0]
print([id2label[i] for i in pred_ids])

# 用原 Trainer.predict 做整體指標
pred_results = trainer.predict(train_ds)
print(pred_results.metrics)

# （選做）用 matplotlib 畫出不同 epoch F1 變化曲線
import matplotlib.pyplot as plt
f1s = [m["eval_f1"] for m in trainer.state.log_history if "eval_f1" in m]
plt.plot(f1s)
plt.title("Epoch vs F1")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.show()
